# NeuroNav SNN MNIST Training Colab


## Purpose

This Colab notebook builds the first publishable baseline for NeuroNav: a reproducible LIF SNN trained on MNIST using surrogate gradients. It deliberately reports only measured notebook outputs. Do not claim FPGA power or timing from this notebook.

In [ ]:
!pip -q install snntorch torch torchvision torchaudio torchmetrics scikit-learn matplotlib pandas

In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import snntorch as snn
from snntorch import surrogate, spikegen
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
# Configuration: keep this cell as the experiment record.
cfg = {
    'dataset': 'MNIST',
    'num_steps': 25,
    'batch_size': 128,
    'epochs': 5,
    'learning_rate': 1e-3,
    'beta': 0.95,
    'hidden_neurons': 128,
    'encoding': 'rate',
    'surrogate': 'atan',
    'seed': SEED,
}
print(json.dumps(cfg, indent=2))

In [ ]:
data_path = '/content/data/mnist'
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize((0,), (1,)),
])

train_ds = datasets.MNIST(data_path, train=True, download=True, transform=transform)
test_ds = datasets.MNIST(data_path, train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False)
print(len(train_ds), len(test_ds))

In [ ]:
class SNNClassifier(nn.Module):
    def __init__(self, hidden_neurons=128, beta=0.95):
        super().__init__()
        spike_grad = surrogate.atan()
        self.fc1 = nn.Linear(28 * 28, hidden_neurons, bias=False)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2 = nn.Linear(hidden_neurons, 10, bias=False)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        spk1_mem = self.lif1.init_leaky()
        spk2_mem = self.lif2.init_leaky()
        spk2_rec = []
        mem2_rec = []
        x_flat = x.view(x.size(0), -1)
        spk_in = spikegen.rate(x_flat, num_steps=cfg['num_steps'])
        for step in range(cfg['num_steps']):
            cur1 = self.fc1(spk_in[step])
            spk1, spk1_mem = self.lif1(cur1, spk1_mem)
            cur2 = self.fc2(spk1)
            spk2, spk2_mem = self.lif2(cur2, spk2_mem)
            spk2_rec.append(spk2)
            mem2_rec.append(spk2_mem)
        return torch.stack(spk2_rec), torch.stack(mem2_rec)

model = SNNClassifier(cfg['hidden_neurons'], cfg['beta']).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(model)
print('parameter_count:', param_count)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'])

def batch_accuracy(spk_rec, labels):
    spike_counts = spk_rec.sum(dim=0)
    pred = spike_counts.argmax(dim=1)
    return (pred == labels).float().mean().item(), pred, spike_counts

history = []
start = time.time()
for epoch in range(cfg['epochs']):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    for data, labels in train_loader:
        data, labels = data.to(device), labels.to(device)
        spk_rec, mem_rec = model(data)
        loss = sum(criterion(mem_rec[step], labels) for step in range(cfg['num_steps']))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc, _, _ = batch_accuracy(spk_rec.detach(), labels)
        total_loss += loss.item()
        total_acc += acc
        n_batches += 1
    row = {'epoch': epoch + 1, 'train_loss': total_loss / n_batches, 'train_acc': total_acc / n_batches}
    history.append(row)
    print(row)
print('training_seconds:', round(time.time() - start, 2))

In [ ]:
model.eval()
all_preds, all_labels = [], []
spike_count_total = 0.0
sample_count = 0
with torch.no_grad():
    for data, labels in test_loader:
        data, labels = data.to(device), labels.to(device)
        spk_rec, mem_rec = model(data)
        _, pred, spike_counts = batch_accuracy(spk_rec, labels)
        all_preds.extend(pred.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        spike_count_total += spike_counts.sum().item()
        sample_count += labels.numel()

acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
avg_output_spikes = spike_count_total / sample_count
cm = confusion_matrix(all_labels, all_preds)
print('test_accuracy:', acc)
print('avg_output_spikes_per_sample:', avg_output_spikes)
print(cm)
print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
# Export measured artifacts for later digital-twin and RTL work.
out_dir = Path('/content/neuro_nav_outputs')
out_dir.mkdir(exist_ok=True)
torch.save({'model_state': model.state_dict(), 'cfg': cfg}, out_dir / 'snn_mnist_float.pt')
metrics = {
    'config': cfg,
    'parameter_count': int(param_count),
    'test_accuracy': acc,
    'avg_output_spikes_per_sample': avg_output_spikes,
    'confusion_matrix': cm.tolist(),
    'history': history,
}
(out_dir / 'metrics_float.json').write_text(json.dumps(metrics, indent=2))
print('saved:', out_dir)

In [ ]:
# Fixed-point export starter: int8 symmetric quantization per layer.
def quantize_tensor_symmetric(t, bits=8):
    qmax = 2 ** (bits - 1) - 1
    scale = float(t.abs().max().item() / qmax) if t.abs().max().item() > 0 else 1.0
    q = torch.clamp(torch.round(t / scale), -qmax - 1, qmax).to(torch.int8)
    return q.cpu().numpy(), scale

exports = {}
scales = {}
for name, tensor in model.state_dict().items():
    q, s = quantize_tensor_symmetric(tensor, bits=8)
    exports[name.replace('.', '_')] = q
    scales[name] = s
np.savez(out_dir / 'weights_int8_symmetric.npz', **exports)
(out_dir / 'quantization_scales.json').write_text(json.dumps(scales, indent=2))
print('exported int8 weights and scales')

## Next measured gates

- If test accuracy is below 90%, tune hidden neurons, time steps, beta, and epochs before hardware work.
- Run the digital-twin notebook using the saved weights.
- Do not report FPGA power or resource utilization from this notebook.